In [ ]:
DATASET = r"Z:\Joseph\250528_B2_003"   # <-- point at your dataset folder

import sys
sys.path.insert(0, r"Z:\Joseph\orgpipe")
from pathlib import Path
from orgpipe import config, layout
ds = Path(DATASET)
cfg = config.load_config(ds)
print("frame rate:", config.resolve_frame_rate(ds, cfg))

In [ ]:
# --- load + organoid split ---
from orgpipe import stage_roi, plots
import matplotlib.pyplot as plt
F, Fneu, stat = stage_roi.load_plane0(ds)
idx, xy, labels = stage_roi.split_organoids(stat)
plots.gmm_scatter(xy, labels, str(ds / "assembloid_demo.jpg"))
print({0: idx[0].size, 1: idx[1].size})

In [ ]:
# --- dF/F z-scores + TUNING: look at these histograms, then set amp_min_z /
# burst_z in <dataset>/orgpipe.json, then re-run the FIRST cell and this one ---
r = cfg["roi"]
dfz = stage_roi.compute_dfz(F, Fneu, r["neuropil_r"], r["baseline_pctl"])
fig = plots.amp_histograms(dfz); plt.show()

In [ ]:
# --- filter + latency-sorted traces ---
import numpy as np
good, order = {}, {}
for k in (0, 1):
    good[k], _, order[k] = stage_roi.select_good(dfz, idx[k], r["amp_min_z"],
                                                 r["burst_z"], r["min_gap_fr"])
    print("organoid", k, ":", good[k].size, "kept")
    fig = plots.trace_stack(dfz, order[k]); plt.show()

In [ ]:
# --- full analysis (same code path as `orgpipe analyze`) ---
bundle = stage_roi.run(ds, cfg)